In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import re
import json
import matplotlib.pyplot as plt

def np_repr_to_arr(s):
  s_with_commas = re.sub(r'([\d\]])\s+(-?[\d[])', r'\1, \2', s.strip())
  return np.array(json.loads(s_with_commas, parse_float=float))

class GenderEmotionDataset(Dataset):
    def __init__(self, df):
        self.emonet_logits = np.stack(df['emonet_logits'].values)
        self.fairface_logits = np.stack(df['fairface_logits'].values)
        
        emotion_map = {'neutral': 0, 'happy': 1, 'sad': 2, 'surprised': 3, 
                      'fear': 4, 'disgust': 5, 'angry': 6, 'contempt': 7}
        
        self.emotion_labels = df['label'].map(emotion_map)
        
        if self.emotion_labels.isna().any():
            print(f"Failed to map emotions: {df.loc[self.emotion_labels.isna(), 'label'].unique()}")
            self.emotion_labels = self.emotion_labels.fillna(0).astype(int).values
        else:
            self.emotion_labels = self.emotion_labels.astype(int).values
        
        gender_map = {'male': 0, 'female': 1}
        
        self.gender_labels = df['original_gender'].map(gender_map)
        
        if self.gender_labels.isna().any():
            print(f"Failed to map genders: {df.loc[self.gender_labels.isna(), 'original_gender'].unique()}")
            self.gender_labels = self.gender_labels.fillna(0).astype(int).values
        else:
            self.gender_labels = self.gender_labels.astype(int).values
        
    def __len__(self):
        return len(self.emotion_labels)
    
    def __getitem__(self, idx):
        return (torch.FloatTensor(self.emonet_logits[idx]), 
                torch.FloatTensor(self.fairface_logits[idx]),
                self.emotion_labels[idx],
                self.gender_labels[idx])

class GenderEmotionModel(nn.Module):
    def __init__(self, use_correction=True, independent_correction=False):
        super().__init__()
        self.use_correction = use_correction
        self.independent_correction = independent_correction
        
        if self.use_correction:
            self.net = nn.Sequential(
                nn.Linear(8 + 2, 24),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(24, 24),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(24, 8 + 2)
            )
            if self.independent_correction:
                self.net1 = nn.Sequential(
                    nn.Linear(8, 24),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(24, 24),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(24, 8)
                )

                self.net2 = nn.Sequential(
                    nn.Linear(2, 24),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(24, 24),
                    nn.ReLU(),
                    nn.Dropout(0.2),
                    nn.Linear(24, 2)
                )
            
    
    def forward(self, emonet_logits, fairface_logits):
        if not self.use_correction:
            return emonet_logits, fairface_logits
        
        combined = torch.cat([emonet_logits, fairface_logits], dim=1)
        if self.independent_correction:
            out1 = self.net1(emonet_logits)
            out2 = self.net2(fairface_logits)
            return out1, out2
        else:
            out = self.net(combined)
        
        return out[..., :8],  out[..., -2:]

In [ ]:
def plot_training_history(history):
    """
    Plots the training and validation loss and accuracy.
    
    Args:
        history (dict): A dictionary containing training and validation metrics for each epoch.
    """
    fig, axs = plt.subplots(3, 1, figsize=(12, 18))
    
    epochs = range(1, len(history['train_loss']) + 1)

    # Plot 1: Loss
    axs[0].plot(epochs, history['train_loss'], 'b-', label='Training Loss')
    axs[0].plot(epochs, history['val_loss'], 'r-', label='Validation Loss')
    axs[0].set_title('Training and Validation Loss')
    axs[0].set_xlabel('Epochs')
    axs[0].set_ylabel('Loss')
    axs[0].legend()
    axs[0].grid(True)

    # Plot 2: Emotion Accuracy
    axs[1].plot(epochs, history['train_emotion_acc'], 'b-', label='Training Emotion Accuracy')
    axs[1].plot(epochs, history['val_emotion_acc'], 'r-', label='Validation Emotion Accuracy')
    axs[1].set_title('Training and Validation Emotion Accuracy')
    axs[1].set_xlabel('Epochs')
    axs[1].set_ylabel('Accuracy')
    axs[1].legend()
    axs[1].grid(True)

    # Plot 3: Gender Accuracy
    axs[2].plot(epochs, history['train_gender_acc'], 'b-', label='Training Gender Accuracy')
    axs[2].plot(epochs, history['val_gender_acc'], 'r-', label='Validation Gender Accuracy')
    axs[2].set_title('Training and Validation Gender Accuracy')
    axs[2].set_xlabel('Epochs')
    axs[2].set_ylabel('Accuracy')
    axs[2].legend()
    axs[2].grid(True)

    plt.tight_layout()
    plt.show()

def train_correction_model(df_train, model, epochs=50, lr=0.001):
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    
    train_df, val_df = train_test_split(df_train, test_size=0.2, stratify=df_train['label'])
    
    train_dataset = GenderEmotionDataset(train_df)
    val_dataset = GenderEmotionDataset(val_df)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    model = model.to(device)
    if not model.use_correction:
        return model, None
    
    emotion_criterion = nn.CrossEntropyLoss()
    gender_criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_emotion_acc': [],
        'val_emotion_acc': [],
        'train_gender_acc': [],
        'val_gender_acc': []
    }
    
    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0
        train_emotion_correct = 0
        train_gender_correct = 0
        train_total = 0

        for emonet_logits, fairface_logits, emotion_labels, gender_labels in train_loader:
            emonet_logits = emonet_logits.to(device)
            fairface_logits = fairface_logits.to(device)
            emotion_labels = emotion_labels.long().to(device)
            gender_labels = gender_labels.long().to(device)
            
            optimizer.zero_grad()
            emotion_out, gender_out = model(emonet_logits, fairface_logits)
            
            emotion_loss = emotion_criterion(emotion_out, emotion_labels)
            gender_loss = gender_criterion(gender_out, gender_labels)
            loss = emotion_loss + gender_loss
            
            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()
            
            _, emotion_pred = torch.max(emotion_out, 1)
            _, gender_pred = torch.max(gender_out, 1)
            train_emotion_correct += (emotion_pred == emotion_labels).sum().item()
            train_gender_correct += (gender_pred == gender_labels).sum().item()
            train_total += emotion_labels.size(0)

        epoch_train_loss = running_train_loss / len(train_loader)
        epoch_train_emotion_acc = train_emotion_correct / train_total
        epoch_train_gender_acc = train_gender_correct / train_total
        
        model.eval()
        running_val_loss = 0.0
        val_emotion_correct = 0
        val_gender_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for emonet_logits, fairface_logits, emotion_labels, gender_labels in val_loader:
                emonet_logits = emonet_logits.to(device)
                fairface_logits = fairface_logits.to(device)
                emotion_labels = emotion_labels.long().to(device)
                gender_labels = gender_labels.long().to(device)
                
                emotion_out, gender_out = model(emonet_logits, fairface_logits)

                emotion_loss = emotion_criterion(emotion_out, emotion_labels)
                gender_loss = gender_criterion(gender_out, gender_labels)
                running_val_loss += (emotion_loss + gender_loss).item()
                
                _, emotion_pred = torch.max(emotion_out, 1)
                _, gender_pred = torch.max(gender_out, 1)
                
                val_emotion_correct += (emotion_pred == emotion_labels).sum().item()
                val_gender_correct += (gender_pred == gender_labels).sum().item()
                val_total += emotion_labels.size(0)
        
        epoch_val_loss = running_val_loss / len(val_loader)
        epoch_val_emotion_acc = val_emotion_correct / val_total
        epoch_val_gender_acc = val_gender_correct / val_total

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_emotion_acc'].append(epoch_train_emotion_acc)
        history['val_emotion_acc'].append(epoch_val_emotion_acc)
        history['train_gender_acc'].append(epoch_train_gender_acc)
        history['val_gender_acc'].append(epoch_val_gender_acc)
        
        # Conditional printing to keep console clean
        if (epoch + 1) % (epochs // 10) == 0 or epoch == epochs - 1:
            print(f'Epoch {epoch+1}/{epochs}: \n'
                  f'  Train Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}\n'
                  f'  Train Emotion Acc: {100*epoch_train_emotion_acc:.1f}%, Val Emotion Acc: {100*epoch_val_emotion_acc:.1f}%\n'
                  f'  Train Gender Acc: {100*epoch_train_gender_acc:.1f}%, Val Gender Acc: {100*epoch_val_gender_acc:.1f}%')
    
    return model, history

def evaluate_models(df_test, model1=None, model2=None):
    device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')
    
    emonet_logits = torch.FloatTensor(np.stack(df_test['emonet_logits'].values)).to(device)
    fairface_logits = torch.FloatTensor(np.stack(df_test['fairface_logits'].values)).to(device)
    
    model1.eval()
    with torch.no_grad():
        baseline_emotion, baseline_gender = model1(emonet_logits, fairface_logits)
    
    emotion_map = {'neutral': 0, 'happy': 1, 'sad': 2, 'surprised': 3, 
                  'fear': 4, 'disgust': 5, 'angry': 6, 'contempt': 7}
    emotion_labels_inv = {v: k for k, v in emotion_map.items()}
    
    df_test['emonet_pred'] = torch.argmax(baseline_emotion.cpu(), dim=1).numpy()
    df_test['emonet_pred'] = df_test['emonet_pred'].map(emotion_labels_inv)
    
    df_test['fairface_pred'] = torch.argmax(baseline_gender.cpu(), dim=1).numpy()
    df_test['fairface_pred'] = df_test['fairface_pred'].map({0: 'male', 1: 'female'})

    model2.eval()
    with torch.no_grad():
        corrected_emotion, corrected_gender = model2(emonet_logits, fairface_logits)
    
    df_test['corrected_emotion'] = torch.argmax(corrected_emotion.cpu(), dim=1).numpy()
    df_test['corrected_emotion'] = df_test['corrected_emotion'].map(emotion_labels_inv)
    
    df_test['corrected_gender'] = torch.argmax(corrected_gender.cpu(), dim=1).numpy()
    df_test['corrected_gender'] = df_test['corrected_gender'].map({0: 'male', 1: 'female'})
    
    return df_test

In [ ]:
%%time

# Train on ADFES
data_adfes = pd.read_csv("../adfes_emonet_fairface.csv")
data_adfes['emonet_logits'] = data_adfes['emonet_logits'].apply(np_repr_to_arr)
data_adfes['fairface_logits'] = data_adfes['fairface_logits'].apply(np_repr_to_arr)

# Evaluate on both datasets
data_faces = pd.read_csv("../faces_emonet_fairface.csv")
data_faces['emonet_logits'] = data_faces['emonet_logits'].apply(np_repr_to_arr)
data_faces['fairface_logits'] = data_faces['fairface_logits'].apply(np_repr_to_arr)

EPOCHS = 500

baseline_model = GenderEmotionModel(use_correction=False, independent_correction=False)
baseline_model, history_baseline = train_correction_model(data_faces, baseline_model, epochs=EPOCHS)

correction_model = GenderEmotionModel(use_correction=True)
correction_model, history = train_correction_model(data_faces, correction_model, epochs=EPOCHS)

In [ ]:
plot_training_history(history)

data_adfes_eval = evaluate_models(data_adfes, baseline_model, correction_model)
data_faces_eval = evaluate_models(data_faces, baseline_model, correction_model)

# Print accuracies
print("\nADFES Results:")
print(f"Emonet Emotion Acc: {100*(data_adfes_eval['emonet_pred'] == data_adfes_eval['label']).mean():.1f}%")
print(f"Fairface Gender Acc: {100*(data_adfes_eval['fairface_pred'] == data_adfes_eval['original_gender']).mean():.1f}%")
print(f"Corrected Emotion Acc: {100*(data_adfes_eval['corrected_emotion'] == data_adfes_eval['label']).mean():.1f}%")
print(f"Corrected Gender Acc: {100*(data_adfes_eval['corrected_gender'] == data_adfes_eval['original_gender']).mean():.1f}%")

print("\nFaces Dataset Results:")
print(f"Emonet Emotion Acc: {100*(data_faces_eval['emonet_pred'] == data_faces_eval['label']).mean():.1f}%")
print(f"Fairface Gender Acc: {100*(data_faces_eval['fairface_pred'] == data_faces_eval['original_gender']).mean():.1f}%")
print(f"Corrected Emotion Acc: {100*(data_faces_eval['corrected_emotion'] == data_faces_eval['label']).mean():.1f}%")
print(f"Corrected Gender Acc: {100*(data_faces_eval['corrected_gender'] == data_faces_eval['original_gender']).mean():.1f}%")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import numpy as np

# --- Configure matplotlib for LaTeX-quality output ---
mpl.rcParams.update({
    'text.usetex': True,
    'font.family': 'serif',
    'font.serif': ['Computer Modern Roman'],
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.titlesize': 14,
})

def calculate_accuracy_metrics(df, true_col, pred_col, group1_col, group2_col):
    """
    Calculates accuracy metrics for a given configuration. This function is unchanged.

    Args:
        df (pd.DataFrame): The dataframe containing the evaluation results.
        true_col (str): The column name of the ground truth labels.
        pred_col (str): The column name of the predicted labels.
        group1_col (str): The first column to group by (e.g., gender).
        group2_col (str): The second column to group by (e.g., emotion).

    Returns:
        dict: A dictionary containing the calculated metrics.
    """
    # Calculate the accuracy matrix
    is_correct = (df[true_col] == df[pred_col])
    accuracy_series = is_correct.groupby([df[group1_col], df[group2_col]], observed=True).mean()
    accuracy_matrix = (accuracy_series * 100).unstack().fillna(0)

    # 1. Calculate Macro Average Accuracy
    macro_avg_acc = accuracy_matrix.values.mean()

    # 2. Calculate gender accuracy gaps, if applicable
    largest_gap = np.nan
    average_gap = np.nan
    
    index_lower = [str(i).lower() for i in accuracy_matrix.index]
    if 'gender' in group1_col.lower() and 'female' in index_lower and 'male' in index_lower:
        female_idx_pos = index_lower.index('female')
        male_idx_pos = index_lower.index('male')
        
        female_accuracies = accuracy_matrix.iloc[female_idx_pos]
        male_accuracies = accuracy_matrix.iloc[male_idx_pos]

        gap_per_emotion = (female_accuracies - male_accuracies).abs()
        largest_gap = gap_per_emotion.max()
        average_gap = gap_per_emotion.mean()

    return {
        'Macro-Avg Accuracy (%)': macro_avg_acc,
        'Largest Gender Gap (%)': largest_gap,
        'Average Gender Gap (%)': average_gap
    }

def plot_paired_accuracy_heatmaps(df, true_col, pred_col_uncorrected, pred_col_corrected, 
                                  group1_col, group2_col, main_title, figure_name, cmap='Blues'):
    """
    Plots a pair of side-by-side transposed heatmaps for 'Uncorrected' vs 'Corrected' results.
    The plots are taller and narrower, sharing a y-axis and a colorbar. Values in the
    corrected panel carry an up/down arrow indicating improvement or degradation
    relative to the uncorrected panel.
    """
    # --- Define a small helper function to create the TRANSPOSED matrix ---
    def _calculate_transposed_matrix(pred_col):
        is_correct = (df[true_col] == df[pred_col])
        # *** CHANGE: Swapped group1_col and group2_col to transpose the matrix ***
        accuracy_series = is_correct.groupby([df[group2_col], df[group1_col]], observed=True).mean()
        matrix = (accuracy_series * 100).unstack().fillna(0)
        return matrix

    # --- Calculate matrices for both scenarios ---
    matrix_uncorrected = _calculate_transposed_matrix(pred_col_uncorrected)
    matrix_corrected = _calculate_transposed_matrix(pred_col_corrected)

    # --- Create the figure with two subplots that share a y-axis ---
    # *** CHANGE: Adjusted figsize logic for taller, narrower plots ***
    fig_height = 4
    fig_width = 4
    
    fig, axes = plt.subplots(1, 2, figsize=(fig_width, fig_height), sharey=True)
    fig.patch.set_alpha(0.0)
    for ax in axes:
        ax.patch.set_alpha(0.0)
    display(main_title)
    # fig.suptitle(main_title, fontsize=16, y=1.02)

    # --- Annotations: plain values on the left, values with directional arrows on the right ---
    annot_uncorrected = np.array([[f'${v:.1f}$' for v in row] for row in matrix_uncorrected.values])

    diff = matrix_corrected.values - matrix_uncorrected.values
    annot_corrected = np.empty(matrix_corrected.shape, dtype=object)
    for i in range(matrix_corrected.shape[0]):
        for j in range(matrix_corrected.shape[1]):
            v = matrix_corrected.values[i, j]
            d = diff[i, j]
            if d > 0:
                annot_corrected[i, j] = f'${v:.1f}' + r'\,\uparrow$'
            elif d < 0:
                annot_corrected[i, j] = f'${v:.1f}' + r'\,\downarrow$'
            else:
                annot_corrected[i, j] = f'${v:.1f}$'

    # --- Common settings for both heatmaps ---
    heatmap_kws = {
        'cmap': cmap,
        'vmin': 0,
        'vmax': 100,
        'linewidths': .5,
        'linecolor': 'lightgray',
        'annot_kws': {'size': 11},
        'fmt': '',  # Required when annot is a string array
    }

    # --- Plot 1: Uncorrected ---
    sns.heatmap(matrix_uncorrected, ax=axes[0], cbar=False, annot=annot_uncorrected, **heatmap_kws)
    axes[0].set_title(r'\textbf{Uncorrected}', fontsize=12)
    # *** CHANGE: Swapped axis labels to match transposed matrix ***
    axes[0].set_ylabel(group2_col.replace('_', ' ').title(), fontsize=11)
    axes[0].set_xlabel(r'Gender', fontsize=11)

    axes[0].tick_params(axis='y', rotation=45)

    # --- Plot 2: Corrected (with the shared colorbar) ---
    cbar_ax = fig.add_axes([.92, .15, .02, .7])
    cbar_ax.patch.set_alpha(0.0)
    sns.heatmap(matrix_corrected, ax=axes[1], cbar=True, cbar_ax=cbar_ax,
                cbar_kws={'label': r'Accuracy / Recall (\%)'},
                annot=annot_corrected, **heatmap_kws)
    axes[1].set_title(r'\textbf{Corrected}', fontsize=12)
    axes[1].set_ylabel('') # Hide redundant y-axis label
    # *** CHANGE: Swapped axis labels to match transposed matrix ***
    axes[1].set_xlabel(r'Gender', fontsize=11)

    plt.tight_layout(rect=[0, 0, .9, 1]) # Adjust layout to make space for the colorbar
    plt.savefig(f"../figures/mat_{figure_name}.pdf", bbox_inches='tight', transparent=True)
    plt.show()


# --- Main Script (Unchanged from previous version) ---

# Assume data_adfes_eval and data_faces_eval are already loaded and available

# --- 1. Calculate and Compile All Metrics into a Table ---

# Define the 8 configurations to analyze
configs = [
    # FACES Dataset
    {'dataset': 'FACES', 'task': 'Gender Prediction', 'correction': 'Uncorrected', 'df': data_faces_eval, 
     'true': 'original_gender', 'pred': 'fairface_pred', 'group1': 'original_gender', 'group2': 'label'},
    {'dataset': 'FACES', 'task': 'Gender Prediction', 'correction': 'Corrected', 'df': data_faces_eval, 
     'true': 'original_gender', 'pred': 'corrected_gender', 'group1': 'original_gender', 'group2': 'label'},
    {'dataset': 'FACES', 'task': 'Emotion Prediction', 'correction': 'Uncorrected', 'df': data_faces_eval, 
     'true': 'label', 'pred': 'emonet_pred', 'group1': 'original_gender', 'group2': 'label'},
    {'dataset': 'FACES', 'task': 'Emotion Prediction', 'correction': 'Corrected', 'df': data_faces_eval, 
     'true': 'label', 'pred': 'corrected_emotion', 'group1': 'original_gender', 'group2': 'label'},
    # ADFES Dataset
    {'dataset': 'ADFES', 'task': 'Gender Prediction', 'correction': 'Uncorrected', 'df': data_adfes_eval, 
     'true': 'original_gender', 'pred': 'fairface_pred', 'group1': 'original_gender', 'group2': 'label'},
    {'dataset': 'ADFES', 'task': 'Gender Prediction', 'correction': 'Corrected', 'df': data_adfes_eval, 
     'true': 'original_gender', 'pred': 'corrected_gender', 'group1': 'original_gender', 'group2': 'label'},
    {'dataset': 'ADFES', 'task': 'Emotion Prediction', 'correction': 'Uncorrected', 'df': data_adfes_eval, 
     'true': 'label', 'pred': 'emonet_pred', 'group1': 'original_gender', 'group2': 'label'},
    {'dataset': 'ADFES', 'task': 'Emotion Prediction', 'correction': 'Corrected', 'df': data_adfes_eval, 
     'true': 'label', 'pred': 'corrected_emotion', 'group1': 'original_gender', 'group2': 'label'},
]

# Calculate metrics for each configuration
all_metrics = []
for config in configs:
    metrics = calculate_accuracy_metrics(
        df=config['df'],
        true_col=config['true'],
        pred_col=config['pred'],
        group1_col=config['group1'],
        group2_col=config['group2']
    )
    # Add identifying information to the metrics dictionary
    metrics['Dataset'] = config['dataset']
    metrics['Task'] = config['task']
    metrics['Correction'] = config['correction']
    all_metrics.append(metrics)

# Create the summary DataFrame
results_df = pd.DataFrame(all_metrics)

# Reorder and set index for a clean, publication-ready table
results_df = results_df.set_index(['Dataset', 'Task', 'Correction'])
# Format floating point numbers to two decimal places
pd.options.display.float_format = '{:,.2f}'.format

print("--- Summary of All Metrics ---")
display(results_df)

# --- 2. Generate and Print LaTeX Code for the Table ---
latex_code = results_df.to_latex(
    float_format="%.2f", 
    na_rep='N/A',
    caption="Summary of macro-average accuracy and gender-based accuracy gaps across datasets and tasks.",
    label="tab:summary_metrics",
    longtable=False
)

print("\n--- LaTeX Code for Summary Table ---")
print(latex_code)

# --- 3. Generate Paired Heatmap Plots ---

print("\n--- Paired Accuracy Heatmaps ---")

# Plot 3: FACES Gender Accuracy
plot_paired_accuracy_heatmaps(
    df=data_faces_eval,
    true_col='original_gender',
    pred_col_uncorrected='fairface_pred',
    pred_col_corrected='corrected_gender',
    group1_col='original_gender',
    group2_col='label',
    main_title='FACES: Gender Accuracy by Emotion',
    figure_name='faces_gender',
    cmap='Blues'
)

# Plot 4: FACES Emotion Accuracy
plot_paired_accuracy_heatmaps(
    df=data_faces_eval,
    true_col='label',
    pred_col_uncorrected='emonet_pred',
    pred_col_corrected='corrected_emotion',
    group1_col='original_gender',
    group2_col='label',
    main_title='FACES: Emotion Accuracy by Gender',
    figure_name='faces_emotion',
    cmap='Reds'
)

# Plot 1: ADFES Gender Accuracy
plot_paired_accuracy_heatmaps(
    df=data_adfes_eval,
    true_col='original_gender',
    pred_col_uncorrected='fairface_pred',
    pred_col_corrected='corrected_gender',
    group1_col='original_gender',
    group2_col='label',
    main_title='ADFES: Gender Accuracy by Emotion',
    figure_name='adfes_gender',
    cmap='Blues'
)

# Plot 2: ADFES Emotion Accuracy
plot_paired_accuracy_heatmaps(
    df=data_adfes_eval,
    true_col='label',
    pred_col_uncorrected='emonet_pred',
    pred_col_corrected='corrected_emotion',
    group1_col='original_gender',
    group2_col='label',
    main_title='ADFES: Emotion Accuracy by Gender',
    figure_name='adfes_emotion',
    cmap='Reds'
)
